#### ***06 — TimesFM (extra)***

Google's **TimesFM** is a decoder-only foundation model for time-series forecasting, pre-trained on ~100B time points. It produces forecasts in **zero-shot mode** — no training, no fine-tuning. We use the **2.0-500m** PyTorch checkpoint (`google/timesfm-2.0-500m-pytorch`), an upgrade over the 1.0-200m model we saw in class (longer context, sharper accuracy).

Same train / test split, same rolling-origin evaluator, same metric set as every other notebook in the project. This closes the comparison: naive → classical (SARIMA/SARIMAX) → deep custom (LSTM) → foundation model.

#### ***Dependency: `timesfm[torch]`***

> Install once (the wheel is large; the checkpoint downloads on first use, ~2 GB).

```bash
pip install "timesfm[torch]"
```

First call to `TimesFm(...)` will pull `google/timesfm-2.0-500m-pytorch` from Hugging Face Hub and cache it under `~/.cache/huggingface`. Subsequent runs are local.

GPU strongly recommended (CUDA on RTX 5070 Ti is the comfortable target). The MPS backend on M1 Pro is not fully supported by TimesFM; on Apple Silicon the model falls back to CPU and takes ~30–90 s per forecast.

#### ***Imports***

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import timesfm
from scripts.evaluation import evaluate_model

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

#### ***Load processed data and split***

In [ ]:
data = pd.read_csv('../data/processed_daily.csv', parse_dates=['date'], index_col='date')

TEST_DAYS = 30
HORIZON   = 7
SEASON    = 7

split_date = data.index.max() - pd.Timedelta(days=TEST_DAYS - 1)
train_y    = data.loc[data.index <  split_date, 'energy_kwh']
test_y     = data.loc[data.index >= split_date, 'energy_kwh']

print(f'Train: {len(train_y)}   Test: {len(test_y)}')

#### ***Load TimesFM 2.0 (500M parameters)***

Instantiation downloads the checkpoint on first call. `horizon_len` fixes the maximum forecast horizon we ever request — 7 here.

In [ ]:
tfm = timesfm.TimesFm(
    hparams=timesfm.TimesFmHparams(
        backend            = 'gpu' if torch.cuda.is_available() else 'cpu',
        per_core_batch_size = 32,
        horizon_len         = HORIZON,
        # Default context length for v2.0 is 512; we pass less than that anyway.
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id='google/timesfm-2.0-500m-pytorch'
    ),
)
print('TimesFM 2.0 loaded.')

#### ***Forecast helper — wrap TimesFM in the project's `forecast_fn` interface***

For **daily** data we pass `freq=[0]` (high frequency family, per the TimesFM convention: 0 = daily/hourly, 1 = weekly/monthly, 2 = quarterly/yearly).

In [ ]:
def timesfm_forecast_fn(history, horizon):
    # TimesFM expects a list of numpy arrays, one per series in the batch.
    context = [history.values.astype(np.float32)]
    point_forecast, _ = tfm.forecast(context, freq=[0])
    return point_forecast[0][:horizon]

#### ***Rolling-origin evaluation***

In [ ]:
windows_t, per_t, sum_t = evaluate_model(timesfm_forecast_fn, train_y, test_y,
                                          horizon=HORIZON, season=SEASON)
print('Per-window metrics:')
print(per_t.round(3).to_string(index=False))

#### ***Final leaderboard — every model in the project***

In [ ]:
# ---- Baselines
def baseline_historical_mean(h, k): return np.full(k, h.mean())
def baseline_last_value(h, k):       return np.full(k, h.iloc[-1])
def baseline_seasonal_naive(h, k):
    last = h.iloc[-SEASON:].values
    return np.tile(last, int(np.ceil(k / SEASON)))[:k]
def baseline_drift(h, k):
    T = len(h)
    return h.iloc[-1] + (h.iloc[-1] - h.iloc[0]) / (T - 1) * np.arange(1, k + 1)

# ---- SARIMA / SARIMAX from the cached grid
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings; warnings.filterwarnings('ignore')

EXOG_COLS = ['temp_mean', 'is_weekend', 'is_holiday']
exog_full = data[EXOG_COLS]
grid = pd.read_csv('../data/sarima_grid_results.csv')

def pick_winner(grid_df, variant):
    sub = grid_df[grid_df['variant'] == variant].dropna(subset=['aic']).sort_values('aic').iloc[0]
    return (int(sub.p), int(sub.d), int(sub.q)), (int(sub.P), int(sub.D), int(sub.Q), int(sub.m))

order_s,  sorder_s  = pick_winner(grid, 'no_exog')
order_sx, sorder_sx = pick_winner(grid, 'exog')

def make_sarima_forecast_fn(order, sorder, exog_full=None):
    def forecast_fn(history, horizon):
        ex_train = exog_full.loc[history.index] if exog_full is not None else None
        future_idx = pd.date_range(history.index[-1] + pd.Timedelta(days=1),
                                   periods=horizon, freq='D')
        ex_future = exog_full.loc[future_idx] if exog_full is not None else None
        m = SARIMAX(history, exog=ex_train, order=order, seasonal_order=sorder,
                    enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False, maxiter=200)
        return m.forecast(steps=horizon, exog=ex_future).values
    return forecast_fn

sarima_fn  = make_sarima_forecast_fn(order_s,  sorder_s,  exog_full=None)
sarimax_fn = make_sarima_forecast_fn(order_sx, sorder_sx, exog_full=exog_full)

# ---- LSTM
import json
from pathlib import Path
from scripts.lstm_train import LSTMForecaster, get_device, FEATURE_COLS, TARGET_COL

LSTM_DIR = Path('../data/lstm')
with open(LSTM_DIR / 'best_config.json') as f:
    best = json.load(f)
device = get_device()

lstm_model = LSTMForecaster(
    n_features  = len(FEATURE_COLS),
    hidden_size = best['config']['hidden_size'],
    num_layers  = best['config']['num_layers'],
    dropout     = best['config']['dropout'],
).to(device)
lstm_model.load_state_dict(torch.load(LSTM_DIR / 'best_model.pt', map_location=device))
lstm_model.eval()

SEQ_LEN    = best['config']['seq_len']
X_mean     = np.asarray(best['norms']['X_mean'], dtype=np.float32)
X_std      = np.asarray(best['norms']['X_std'],  dtype=np.float32)
Y_MEAN     = best['norms']['y_mean']
Y_STD      = best['norms']['y_std']
TARGET_IDX = FEATURE_COLS.index(TARGET_COL)

def lstm_forecast_fn(history, horizon):
    last_date    = history.index[-1]
    past_dates   = pd.date_range(end=last_date, periods=SEQ_LEN, freq='D')
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=horizon, freq='D')
    past_features = data.loc[past_dates,   FEATURE_COLS].values.astype(np.float32)
    future_exog   = data.loc[future_dates, FEATURE_COLS].values.astype(np.float32)
    window = (past_features - X_mean) / X_std
    preds  = []
    with torch.no_grad():
        for i in range(horizon):
            x      = torch.from_numpy(window[np.newaxis]).to(device)
            y_norm = lstm_model(x).item()
            y_pred = y_norm * Y_STD + Y_MEAN
            preds.append(y_pred)
            nf              = future_exog[i].copy()
            nf[TARGET_IDX]  = y_pred
            nfn             = (nf - X_mean) / X_std
            window          = np.concatenate([window[1:], nfn[np.newaxis]], axis=0)
    return np.asarray(preds, dtype=float)

# ---- Full leaderboard
models = {
    'Historical mean':                  baseline_historical_mean,
    'Last value':                       baseline_last_value,
    'Seasonal naive':                   baseline_seasonal_naive,
    'Drift':                            baseline_drift,
    f'SARIMA {order_s}x{sorder_s}':     sarima_fn,
    f'SARIMAX {order_sx}x{sorder_sx}':  sarimax_fn,
    f'LSTM {best["name"]}':             lstm_forecast_fn,
    'TimesFM 2.0-500m (zero-shot)':     timesfm_forecast_fn,
}

summaries = {}
windows_by_model = {}
for name, fn in models.items():
    w, _, summary = evaluate_model(fn, train_y, test_y, horizon=HORIZON, season=SEASON)
    summaries[name] = summary
    windows_by_model[name] = w

summary_df = pd.DataFrame(summaries).T

def fmt(row, m):
    return f'{row[f"{m}_mean"]:.2f} ± {row[f"{m}_std"]:.2f}'

leaderboard = pd.DataFrame({
    'MAE':   summary_df.apply(lambda r: fmt(r, 'MAE'),   axis=1),
    'RMSE':  summary_df.apply(lambda r: fmt(r, 'RMSE'),  axis=1),
    'MAPE':  summary_df.apply(lambda r: f'{r["MAPE_mean"]*100:5.1f}% ± {r["MAPE_std"]*100:4.1f}%',   axis=1),
    'sMAPE': summary_df.apply(lambda r: f'{r["sMAPE_mean"]*100:5.1f}% ± {r["sMAPE_std"]*100:4.1f}%', axis=1),
    'MASE':  summary_df.apply(lambda r: fmt(r, 'MASE'),  axis=1),
    'Bias':  summary_df.apply(lambda r: f'{r["Bias_mean"]:+.2f}',  axis=1),
    'R²':    summary_df.apply(lambda r: f'{r["R2_mean"]:+.2f}',    axis=1),
})
leaderboard['_mase'] = summary_df['MASE_mean']
leaderboard = leaderboard.sort_values('_mase').drop(columns='_mase')
leaderboard

#### ***Forecasts vs actuals — top three models***

In [ ]:
# Pick the top-3 by MASE
top3 = summary_df.sort_values('MASE_mean').index[:3].tolist()
print('Top-3 by MASE:', top3)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(test_y.index, test_y.values, 'b-', lw=1.8, label='Actual')

colors = ['darkorange', 'green', 'red']
for name, color in zip(top3, colors):
    idx   = np.concatenate([w['index']       for w in windows_by_model[name]])
    preds = np.concatenate([w['predictions'] for w in windows_by_model[name]])
    ax.plot(idx, preds, '--', color=color, lw=1.2, label=name)

for w in windows_by_model[top3[0]]:
    ax.axvline(w['index'][0], color='grey', alpha=0.3, lw=0.8)

ax.set_xlabel('Date'); ax.set_ylabel('Energy (kWh)')
ax.set_title('Top-3 models by MASE — rolling-origin forecasts')
ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

#### ***MAE per window — visual bar chart***

In [ ]:
mae_long = pd.DataFrame({
    name: [summary_df.loc[name, 'MAE_mean']] for name in models.keys()
}).T.rename(columns={0: 'MAE_mean'})

# Per-window matrix for the heat-bar visual
import itertools
per_window_mae = pd.DataFrame({
    name: [np.mean(np.abs(w['actuals'] - w['predictions'])) for w in windows_by_model[name]]
    for name in models.keys()
}, index=[f'W{i+1}' for i in range(len(next(iter(windows_by_model.values()))))]).T

fig, ax = plt.subplots(figsize=(10, 5))
per_window_mae.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_xlabel('Model'); ax.set_ylabel('MAE (kWh)')
ax.set_title('Per-window MAE across all models')
plt.xticks(rotation=20, ha='right')
plt.tight_layout(); plt.show()

#### ***Summary***

The leaderboard above is the headline output of the project: **eight models, four 7-day windows, seven metrics**, all measured with the same evaluator on the same series.

Key questions to discuss in the report:

- **Does any model beat MASE < 0.67?** That was the SARIMAX bar from notebook `04`.
- **Does any model achieve R² > 0?** Positive R² is the first sign that we are explaining test-period variance rather than just predicting around the mean.
- **TimesFM is zero-shot** — it never saw this dataset. Its position in the leaderboard tells us how much a generic foundation model brings out-of-the-box vs. a model fitted (SARIMAX) or trained (LSTM) on this specific series.

That's the story the comparison is designed to surface.